<a href="https://colab.research.google.com/github/poonam19-ui/nsbi-lhc-toolkit/blob/ml4hep_school_tutorial/workshops/ml4hep_tifr_colab/Exercise_1_summary_statistics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/livaage/nsbi-lhc-toolkit/blob/ml4hep-tifr-colab/workshops/ml4hep_tifr_colab/Exercise_1_summary_statistics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 1 — Summary statistics and likelihood fits

**Google Colab fallback.** Run the setup cell below **first** — it installs the
dependencies, pulls the `nsbi_common_utils` package plus the tutorial helpers
(`utils.py`, `generate_distributions.py`), generates the dataset, and moves into
the tutorial directory so everything below runs unchanged.

> 💡 For the training notebooks, switch to a **GPU runtime**
> (*Runtime → Change runtime type → GPU*) and consider lowering `number_of_epochs`
> / `N_TRAIN` for a quick pass.


In [ ]:
# ============================================================================
# Google Colab setup — run me first.  Safe to re-run; a no-op off Colab.
# ============================================================================
import os, sys

# --- config -----------------------------------------------------------------
REPO_URL = "https://github.com/livaage/nsbi-lhc-toolkit.git"  # package + tutorial helpers
BRANCH   = "ml4hep_school_tutorial"
N_BKG, N_SIG = 5_000_000, 5000_000     # Colab-sized dataset (raise for less MC noise in the fit)
USE_DRIVE = True                   # True -> save data/models to Google Drive so they
                                    # persist across notebooks & sessions (see notes above)
# ----------------------------------------------------------------------------

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        ROOT = "/content/drive/MyDrive/ml4hep_tifr_colab"
    else:
        ROOT = "/content"
    os.makedirs(ROOT, exist_ok=True)
    os.chdir(ROOT)

    # 1) fetch ONLY the package source + tutorial helpers (skip Git-LFS / big blobs)
    if not os.path.isdir("nsbi-lhc-toolkit"):
        os.environ["GIT_LFS_SKIP_SMUDGE"] = "1"
        !git clone --depth 1 --filter=blob:none --sparse --branch $BRANCH $REPO_URL
        !cd nsbi-lhc-toolkit && git sparse-checkout set src workshops/ml4hep_tifr

    # 2) make `import nsbi_common_utils` work (pure-python src layout, no build step)
    src = os.path.abspath("nsbi-lhc-toolkit/src")
    if src not in sys.path:
        sys.path.insert(0, src)

    # 3) runtime deps Colab doesn't already ship (torch/jax/sklearn/... are preinstalled)
    !pip install -q pytorch-lightning onnx onnxruntime onnxscript iminuit mplhep

    # 4) work from the tutorial dir so utils.py / generate_distributions.py and the
    #    ./dataframes, ./models_* relative paths resolve just like a local run
    os.chdir("nsbi-lhc-toolkit/workshops/ml4hep_tifr")

    # 5) generate the Gaussian-mixture samples if they aren't there yet
    if not os.path.exists("dataframes/signal.parquet"):
        !python generate_distributions.py --n_bkg $N_BKG --n_sig $N_SIG

print("Working dir:", os.getcwd())


Mounted at /content/drive
Cloning into 'nsbi-lhc-toolkit'...
remote: Enumerating objects: 32, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 32 (delta 0), reused 21 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (32/32), 6.51 KiB | 740.00 KiB/s, done.
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 9 (delta 0), reused 6 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (9/9), 131.84 KiB | 2.75 MiB/s, done.
remote: Enumerating objects: 29, done.
remote: Counting objects: 100% (29/29), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 29 (delta 2), reused 6 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (29/29), 1.42 MiB | 8.88 MiB/s, done.
Resolving deltas: 100% (2/2), done.
Updating files: 100% (38/38), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 6.0 MB/s e

# Exercise 1 — Summary statistics and likelihood fits

The goal of this first exercise is to start with the most familiar HEP workflow: compress simulated events into binned observables, build a likelihood for the signal strength, and compare how much information different summaries retain.

By the end of the notebook, students should be able to explain three ideas:

1. A histogram is a **summary statistic**: it keeps some information from the event sample and discards the rest.
2. A binned Poisson likelihood can be used to fit the signal strength $\mu$.
3. The optimal one-dimensional summary for testing signal against background is related to the density ratio $p_S(x)/p_B(x)$.

In [ ]:
import os
import nsbi_common_utils
import pandas as pd
import numpy as np
import mplhep as hep
import pickle
import matplotlib.pyplot as plt
from nsbi_common_utils.training.utils import load_trained_model
import pprint

## Lecture mode

This version keeps the original computational workflow intact, but adds short theory checkpoints and reveal/hide controls for presenting the exercise live.  The result-producing cells are marked to open with outputs hidden; use the buttons below to reveal them when you reach that point in the lecture.

In [ ]:
from IPython.display import HTML, display

# Lecture controls: run this cell if the buttons are not visible after opening the notebook.
display(HTML(r"""
<div class="lecture-output-controller" style="border:1px solid #bbb; border-radius:8px; padding:12px; margin:8px 0; background:#fafafa;">
  <b>Lecture controls</b><br>
  Use these buttons while presenting to hide or reveal notebook results.
  The computational cells themselves are unchanged.
  <div style="margin-top:8px; display:flex; gap:8px; flex-wrap:wrap;">
    <button style="padding:6px 10px;" onclick="
      (function(){
        const selectors = ['.jp-OutputArea', 'div.output_wrapper', 'div.output', '.cell-output'];
        document.querySelectorAll(selectors.join(',')).forEach(function(el){
          if (el.querySelector && el.querySelector('.lecture-output-controller')) {
            el.style.setProperty('display', 'block', 'important');
          } else {
            el.style.setProperty('display', 'none', 'important');
          }
        });
        document.querySelectorAll('.jp-Cell').forEach(function(cell){
          if (!(cell.querySelector && cell.querySelector('.lecture-output-controller'))) {
            cell.classList.add('jp-mod-outputsHidden');
          }
        });
      })();
    ">Hide results</button>
    <button style="padding:6px 10px;" onclick="
      (function(){
        const selectors = ['.jp-OutputArea', 'div.output_wrapper', 'div.output', '.cell-output'];
        document.querySelectorAll(selectors.join(',')).forEach(function(el){
          el.style.setProperty('display', 'block', 'important');
        });
        document.querySelectorAll('.jp-Cell').forEach(function(cell){
          cell.classList.remove('jp-mod-outputsHidden');
        });
      })();
    ">Show results</button>
    <button style="padding:6px 10px;" onclick="
      (function(){
        const visible = Array.from(document.querySelectorAll('.jp-OutputArea, div.output_wrapper, div.output, .cell-output'))
          .filter(function(el){ return !(el.querySelector && el.querySelector('.lecture-output-controller')); })
          .some(function(el){ return getComputedStyle(el).display !== 'none'; });
        const target = visible ? 'none' : 'block';
        const selectors = ['.jp-OutputArea', 'div.output_wrapper', 'div.output', '.cell-output'];
        document.querySelectorAll(selectors.join(',')).forEach(function(el){
          if (el.querySelector && el.querySelector('.lecture-output-controller')) {
            el.style.setProperty('display', 'block', 'important');
          } else {
            el.style.setProperty('display', target, 'important');
          }
        });
        document.querySelectorAll('.jp-Cell').forEach(function(cell){
          if (!(cell.querySelector && cell.querySelector('.lecture-output-controller'))) {
            if (target === 'none') { cell.classList.add('jp-mod-outputsHidden'); }
            else { cell.classList.remove('jp-mod-outputsHidden'); }
          }
        });
      })();
    ">Toggle results</button>
  </div>
  <small style="display:block; margin-top:8px; color:#555;">
    Tip: all existing result-producing cells are also marked as hidden-output cells, so JupyterLab/Notebook should open them collapsed.
  </small>
</div>
"""))


In [ ]:
BASE_PATH = "./dataframes/"

## Samples and event weights

The MC samples are not just unweighted point clouds.  Each event carries a weight, and the sum of the weights represents the expected yield of that process.  In this exercise, histograms are therefore histograms of **expected events**, not simply counts.

For a process $a\in\{S,B\}$, a weighted histogram estimates

\begin{equation}
\nu_i^{(a)} \approx \sum_{e\in a} w_e\,\mathbf{1}(x_e\in\text{bin }i),
\end{equation}

where $\nu_i^{(a)}$ is the expected number of signal or background events in bin $i$.

In [ ]:
signal = pd.read_parquet(f"{BASE_PATH}/signal.parquet")
background = pd.read_parquet(f"{BASE_PATH}/background.parquet")

# Summary statistics

We will start by doing a traditional binned SBI

We will compare two binned analysis: one using a one-dimensional ($x_1$) and one using a two-dimensional observable ($x_1,x_2$).

## From events to summary statistics

A summary statistic is a lower-dimensional representation of the full dataset.  Here the first summaries are:

\begin{equation}
T_1(\mathcal{D}) = \text{histogram of } x_1,
\end{equation}

and

\begin{equation}
T_2(\mathcal{D}) = \text{2D histogram of } (x_1,x_2).
\end{equation}

The one-dimensional histogram is easy to visualize and fit, but it ignores correlations with the other observables.  The two-dimensional histogram keeps more information, but it also needs more bins and therefore more MC statistics.

In [ ]:
def hist_template(values, edges, weights):
    values_clipped = np.clip(values, edges[0], edges[-1])
    hist, _ = np.histogram(values_clipped, bins=edges, weights=weights)
    return hist

var = 'x1'
bins = np.linspace(-3,8,12)

signal_weights = np.array(signal.weight)
background_weights = np.array(background.weight)

h_sig = hist_template(signal[var], bins, signal_weights)
h_bkg = hist_template(background[var], bins, background_weights)

def hist_template_2d(x_vals, y_vals, x_edges, y_edges, weights):

    x_clipped = np.clip(x_vals, x_edges[0], x_edges[-1])
    y_clipped = np.clip(y_vals, y_edges[0], y_edges[-1])

    h2d, _, _ = np.histogram2d(
        x_clipped,
        y_clipped,
        bins=[x_edges, y_edges],
        weights=weights
    )

    return h2d

xvar = "x1"
yvar = "x2"
h2d_sig = hist_template_2d(signal[xvar], signal[yvar], bins, bins, signal_weights)
h2d_bkg = hist_template_2d(background[xvar], background[yvar], bins, bins, background_weights)

<details>
<summary><b>Lecture prompt: what information is lost by binning?</b></summary>

Binning forgets the exact event locations inside each bin.  It also forgets all variables that are not included in the histogram.  This is why $x_1$ alone may give a weaker constraint than $(x_1,x_2)$, and why a learned density-ratio statistic can do even better.

</details>

# Visualizing the statistics

Both summary statistics can be used to perform inference, but neither are sufficient.

Plotting can be a powerful way to show to which degree an inference on the signal strength can be performed.

## Reading the first histogram

The upper panel shows the signal and background expected yields.  Because the signal yield is much smaller than the background yield, the lower panel shows the ratio $S/B$, which is often more informative than the raw stacked histogram.

A bin is statistically useful when it has either a large signal excess or a distinctive signal-to-background pattern.  In practice, both the separation and the background statistics matter.

In [ ]:
def plot_histograms(signal_yield_per_bin, background_yield_per_bin, bins, log_y=False, ax_title=r"$x_1$"):
    fig, (ax, rax) = plt.subplots(
        2, 1,
        figsize=(8.0, 7.0),
        sharex=True,
        gridspec_kw={"height_ratios": [3, 1], "hspace": 0.05},
    )

    # Background drawn as a filled stack (a list, so more backgrounds can be
    # added later). Signal is overlaid UNSTACKED as a step line -- with a true
    # S/B ~ 1e-3 a stacked signal would be an invisible sliver on top.
    hep.histplot(
        [signal_yield_per_bin,background_yield_per_bin],
        bins=bins,
        stack=True,
        histtype="fill",
        label=["Signal","Background"],
        ax=ax,
    )

    centers = 0.5 * (bins[1:] + bins[:-1])
    half_widths = 0.5 * np.diff(bins)

    observation_per_bin = signal_yield_per_bin + background_yield_per_bin
    obs_err = np.sqrt(np.maximum(observation_per_bin, 0.0))

    # Ratio panel: signal / background, with sqrt(obs) propagated as the error.
    prediction = background_yield_per_bin
    ratio = np.divide(
        signal_yield_per_bin, prediction,
        out=np.full_like(observation_per_bin, np.nan, dtype=float),
        where=prediction > 0,
    )
    ratio_err = np.divide(
        obs_err, prediction,
        out=np.full_like(obs_err, np.nan, dtype=float),
        where=prediction > 0,
    )

    rax.errorbar(
        centers, ratio,
        xerr=half_widths, yerr=0,
        fmt="o", markersize=4, color="black",
    )
    # Auto-scale the ratio panel to the finite points (fixed 0-10 hides a ~1e-3 ratio).
    finite = ratio[np.isfinite(ratio)]
    if finite.size:
        rax.set_ylim(0.0, 1.2 * finite.max())

    ax.set_ylabel("Events / bin")
    rax.set_ylabel("Signal / Bkg")
    rax.set_xlabel(ax_title)
    ax.legend(fontsize=11)

    if log_y:
        positive = np.concatenate([background_yield_per_bin, signal_yield_per_bin])
        positive = positive[positive > 0]
        if positive.size:
            ax.set_yscale("log")
            ax.set_ylim(max(positive.min() * 0.2, 1.0e-3), positive.max() * 20.0)
    else:
        ymax = max(
            np.max(background_yield_per_bin) if background_yield_per_bin.size else 0.0,
            np.max(signal_yield_per_bin) if signal_yield_per_bin.size else 0.0,
            np.max(observation_per_bin + obs_err) if observation_per_bin.size else 0.0,
        )
        ax.set_ylim(0.0, ymax if ymax > 0 else 1.0)


plot_histograms(h_sig, h_bkg, bins, log_y=True)


## Why look at two dimensions?

The 2D view checks whether signal and background separate through correlations.  If signal and background differ mostly by a correlated deformation, then no single one-dimensional projection will show the full discrimination power.

This is the first hint that the likelihood wants the joint distribution, not only marginal histograms.

In [ ]:
def plot_histograms_2d(signal_yield_per_bin, background_yield_per_bin, bins):

    fig, axs = plt.subplots(1,2,figsize=(10.0, 4.0))

    mesh = axs[0].pcolormesh(
        bins,
        bins,
        background_yield_per_bin.T,
    )
    mesh = axs[1].pcolormesh(
        bins,
        bins,
        signal_yield_per_bin.T,
    )

    fig.colorbar(mesh, ax=axs[0])
    fig.colorbar(mesh, ax=axs[1])
    axs[0].set_title('Background')
    axs[1].set_title('Signal')

    for ax in axs:
        ax.set_xlabel(r"$x_1$")
        ax.set_ylabel(r"$x_2$")
plot_histograms_2d(h2d_sig,h2d_bkg,bins)

## Turning summaries into a statistical model

Once we have expected signal and background yields per bin, the statistical model is simple:

\begin{equation}
\lambda_i(\mu) = \mu\,\nu_i^{(S)} + \nu_i^{(B)}.
\end{equation}

The parameter $\mu$ scales the nominal signal yield.  The pseudo-observation used here is the Asimov-style expectation $N_i=\nu_i^{(S)}+\nu_i^{(B)}$, so the ideal answer is close to $\mu=1$, up to MC/statistical fluctuations in the constructed sample.

# Workspaces

Workspaces provide a serialization of the information used to perform inference.

## Workspace representation

The workspace is a serializable description of the likelihood.  It stores:

- the signal template $\nu_i^{(S)}$,
- the background template $\nu_i^{(B)}$,
- the observed bin contents $N_i$,
- the parameter of interest $\mu$.

This is intentionally close to the way many HEP statistical models are exchanged: the inference engine should not need to know how the histograms were produced.

In [ ]:
def make_workspace(signal_yield_per_bin,background_yield_per_bin,observation_yield_per_bin):
    return {"channels": [
        {
            "name": "SR",
            "samples": [
                {
                    "name": "signal",
                    "data": signal_yield_per_bin,
                    "modifiers": [{"name": "mu", "type": "normfactor", "data": None}],
                },
                {
                    "name": "background",
                    "data": background_yield_per_bin,
                    "modifiers": [],
                },
            ],
            "type": "binned",
        }
    ],
    "measurements": [
        {
            "name": "meas",
            "config": {
                "poi": "mu",
                "parameters": [
                    {"bounds": [[-10.0, 10.0]], "inits": [1.0], "name": "mu"},
                ],
            },
        }
    ],
    "version": "1.0.0",
}

ws_x1 = make_workspace((h_sig).tolist(), (h_bkg).tolist(), (h_sig+h_bkg).tolist())
pprint.pprint(ws_x1)

sig_2dyields = h2d_sig.ravel(order="C")
bkg_2dyields = h2d_bkg.ravel(order="C")
ws_x1x2 = make_workspace((sig_2dyields).tolist(), (bkg_2dyields).tolist(), (sig_2dyields+bkg_2dyields).tolist())


## Fitting the signal strength

The fit minimizes the negative log-likelihood.  For fixed templates and one free parameter, the optimizer searches for the value of $\mu$ whose expected bin counts best describe the observation.

The profile scan below turns this into a curve:

\begin{equation}
t_\mu = -2\log\frac{L(\mu)}{L(\hat\mu)}.
\end{equation}

A narrower curve means the summary statistic gives a stronger constraint on $\mu$.

In [ ]:
model_x1 = nsbi_common_utils.models.sbi_parametric_model(workspace=ws_x1, measurement_to_fit="meas")
list_params, init_values = model_x1.get_model_parameters()
num_unconstrained = model_x1.num_unconstrained_param
inf_model_x1 = nsbi_common_utils.inference.inference(
    model_nll=model_x1.model,
    initial_values=init_values,
    list_parameters=list_params,
    num_unconstrained_params=num_unconstrained,
    model_grad=model_x1.model_grad)

In [ ]:
print("\n" + "="*40)
print(" MODEL X1 FIT RESULTS ")
print("="*40 + "\n")
inf_model_x1.perform_fit(freeze_params=[])

In [ ]:
scan_points_model_x1, NLL_value_model_x1, scan_points_StatOnly_model_x1, NLL_value_StatOnly_model_x1 = inf_model_x1.perform_profile_scan(parameter_name = 'mu',
                             freeze_params = [],
                             bound_range = (0, 3.0),
                             fit_strategy = 0,
                             doStatOnly = True,
                             size = 50)

<details>
<summary><b>Lecture reveal: what should the first scan show?</b></summary>

The scan should have a minimum near the fitted value $\hat\mu$.  If the pseudo-observation is signal plus background, the minimum should be near $\mu=1$.  The width of the curve is the important teaching point: it quantifies how much the chosen statistic, here only $x_1$, can constrain the signal strength.

</details>

In [ ]:
plt.plot(scan_points_model_x1, NLL_value_model_x1, color='red')
plt.axis(ymin=0)
plt.xlabel(r"$\mu_\mathrm{signal}$")
plt.ylabel(r"$t_\mu$")

## Deriving the binned likelihood

For independent bins, the total likelihood is a product of Poisson terms:

\begin{equation}
L(\mu)=\prod_i \mathrm{Pois}\left(N_i\mid \mu\nu_i^{(S)}+\nu_i^{(B)}\right).
\end{equation}

Dropping constants that do not depend on \(\mu\), the negative log-likelihood is

\begin{equation}
-2\log L(\mu)
=2\sum_i\left[\lambda_i(\mu)-N_i\log\lambda_i(\mu)\right]+\text{const.}
\end{equation}

The next cell implements this directly in JAX.  It is useful pedagogically because it shows that the workspace fit is just evaluating this likelihood in a structured way.

## Understanding the math

The probability model for each bin is:

\begin{equation}
p_i = \text{Poisson}(N_i|\lambda_i(\mu))=e^{-(\mu\nu_i^{(S)}+\nu_i^{(B)})}(\mu\nu_i^{(S)}+\nu_i^{(B)})^{N_i}, \qquad i=1, \ldots, n_{\text{bins}}
\end{equation}

and

\begin{equation}
p(\mathcal{D})=\prod_{i=1}^{n_{\text{bins}}} p_i
\end{equation}

In [ ]:
import jax
import jax.numpy as jnp

def nll_manual(params):
    mu = params[0]
    per_bin = -(mu * h_sig + h_bkg)+(h_sig + h_bkg)*jnp.log(mu * h_sig + h_bkg)
    return -2.0 * jnp.sum(per_bin)


inf_manual = nsbi_common_utils.inference.inference(
    model_nll=jax.jit(nll_manual),
    initial_values=[1.0],
    list_parameters=["mu"],
    num_unconstrained_params=1,
    model_grad=jax.jit(jax.grad(nll_manual)),  # analytic gradient, for free
)
inf_manual.perform_fit(freeze_params=[])

## What would the ideal statistic use?

The binned $x_1$ model is intentionally simple, but we know the toy generator.  That means we can evaluate the analytic signal and background densities and construct the truth density ratio

\begin{equation}
r(x)=\frac{p_S(x)}{p_B(x)}.
\end{equation}

This is not how real data works, but it gives a benchmark: it separates the statistical/inference question from the machine-learning approximation question.

In [ ]:
from utils import background_components, signal_components, mixture_density

print(len(background))
print(len(signal))
truth_features = ["z1", "z2", "z3", "z4", "z5"]

# Analytic (truth) density ratios, using the same reference as the training:
asimov_dataset = pd.concat([background, signal], ignore_index=True).astype('float32').copy()
weights_asimov = np.array(asimov_dataset.weight)

PATH_TO_WEIGHTS = "saved_weights/asimov_weight_array.npy"
os.makedirs(os.path.dirname(PATH_TO_WEIGHTS), exist_ok=True)
np.save(PATH_TO_WEIGHTS, weights_asimov)

lam_bkg = background.weight.sum()
lam_sig = signal.weight.sum()
X_asimov = asimov_dataset[truth_features].values
p_bkg = mixture_density(X_asimov, background_components())
p_sig = mixture_density(X_asimov, signal_components())

PATH_TO_PROB_TRUTH = {
    "signal": "saved_prob_asimov/ratio_Sig_truth.npy",
    "background": "saved_prob_asimov/ratio_Bkg_truth.npy",
}
os.makedirs(os.path.dirname(PATH_TO_PROB_TRUTH['signal']), exist_ok=True)

np.save(PATH_TO_PROB_TRUTH["signal"], p_sig)
np.save(PATH_TO_PROB_TRUTH["background"], p_bkg)

## Visualizing the truth ratio

Instead of histogramming $x_1$, we now histogram the truth density ratio.  Events with large $p_S(x)/p_B(x)$ live in regions that are more signal-like.

This compression is much closer to the optimal summary for measuring the amount of signal in a background-dominated sample.

In [ ]:
bins_truth = np.linspace(0,15,15)
count_events = len(signal_weights)
h_sig_truth = hist_template(p_sig[count_events:]/p_bkg[count_events:], bins_truth, signal_weights)
h_bkg_truth = hist_template(p_sig[:count_events]/p_bkg[:count_events], bins_truth, background_weights)
plot_histograms(h_sig_truth,h_bkg_truth,bins_truth, True, r'Truth $p_S/p_B$')

In [ ]:
ws_truth = {
    "channels": [{
        "name": "SR",
        "type": "unbinned",
        "weights": PATH_TO_WEIGHTS,
        "samples": [
            {"name": "signal",
             "data": [lam_sig],
             "ratios": PATH_TO_PROB_TRUTH["signal"],
             "modifiers": [{"name": "mu", "type": "normfactor"}]},
            {"name": "background",
             "data": [lam_bkg],
             "ratios": PATH_TO_PROB_TRUTH["background"],
             "modifiers": []},
        ],
    }],
    "measurements": [{
        "name": "meas",
        "config": {
            "poi": "mu",
            "parameters": [
                {"bounds": [[0, 10]], "inits": [1], "name": "mu"},
            ],
        },
    }],
    "version": "1.0.0",
}

In [ ]:
model_truth = nsbi_common_utils.models.sbi_parametric_model(workspace=ws_truth, measurement_to_fit="meas")
list_params_t, init_values_t = model_truth.get_model_parameters()
inf_truth = nsbi_common_utils.inference.inference(
    model_nll=model_truth.model,
    initial_values=init_values_t,
    list_parameters=list_params_t,
    num_unconstrained_params=model_truth.num_unconstrained_param,
    model_grad=model_truth.model_grad,
)

<details>
<summary><b>Lecture note: why can the fitted minimum shift?</b></summary>

Even for a perfectly specified likelihood, a finite pseudo-dataset does not have to give exactly $\hat\mu=1$.  The fitted value fluctuates because the realized sample fluctuates.  The important comparison is therefore not only the location of the minimum, but also the curvature and the relative improvement between summaries.

</details>

A shift in the maximum likelhood estimate is expected, this is simply the statistical fluction of the sample we generated

In [ ]:
scan_truth, tmu_truth = inf_truth.perform_profile_scan(
    parameter_name="mu", freeze_params=[], bound_range=(0.0, 3.0), fit_strategy=0, size=50)

fig, ax = plt.subplots(figsize=(7, 5))
#ax.plot(scan_pred, tmu_pred, lw=2, label="Approximated (NN) ratios")
ax.plot(scan_truth, tmu_truth, lw=2, ls="--", color='orange', label="Truth")
ax.plot(scan_points_model_x1, NLL_value_model_x1, color='red', label=r'$x_1$ model')
ax.axvline(1.0, color="black", ls=":", lw=1, alpha=0.6)
ax.set_ylim(bottom=0)
ax.set_xlabel(r"$\mu_\mathrm{signal}$")
ax.set_ylabel(r"$t_\mu$")
ax.legend()

## Adding dimensions: more information, more bins

Moving from $x_1$ to $(x_1,x_2)$ should keep more information about the joint signal/background structure.  However, binned methods pay a price: the number of bins grows quickly with dimension.

For example, 10 bins per dimension means

\begin{equation}
10^1=10\quad\text{bins in 1D},\qquad 10^2=100\quad\text{bins in 2D},\qquad 10^5=100000\quad\text{bins in 5D}.
\end{equation}

This is the curse of dimensionality in histogram-based inference.

## Increasing the dimensionality

A 2D summary statistics will have more information.

In [ ]:
model_x1x2 = nsbi_common_utils.models.sbi_parametric_model(workspace=ws_x1x2, measurement_to_fit="meas")
list_params, init_values = model_x1x2.get_model_parameters()
num_unconstrained = model_x1x2.num_unconstrained_param
inf_model_x1x2 = nsbi_common_utils.inference.inference(
    model_nll=model_x1x2.model,
    initial_values=init_values,
    list_parameters=list_params,
    num_unconstrained_params=num_unconstrained,
    model_grad=model_x1x2.model_grad)

In [ ]:
print("\n" + "="*40)
print(" MODEL X1-X2 FIT RESULTS ")
print("="*40 + "\n")
inf_model_x1x2.perform_fit(freeze_params=[])

In [ ]:
scan_points_model_x1x2, NLL_value_model_x1x2, scan_points_StatOnly_model_x1x2, NLL_value_StatOnly_model_x1x2 = inf_model_x1x2.perform_profile_scan(parameter_name = 'mu',
                             freeze_params = [],
                             bound_range = (0, 3.0),
                             fit_strategy = 0,
                             doStatOnly = True,
                             size = 50)

<details>
<summary><b>Lecture reveal: interpreting the 1D vs 2D comparison</b></summary>

If the 2D curve is narrower than the 1D curve, then $(x_1,x_2)$ carries additional information about $\mu$.  If it is still wider than the truth-ratio curve, then there is information in the remaining variables or in the continuous event positions that the 2D histogram did not capture.

</details>

In [ ]:
plt.plot(scan_points_model_x1, NLL_value_model_x1, label=r'$x_1$ model', color='red')
plt.plot(scan_points_model_x1x2, NLL_value_model_x1x2, label=r'$x_1-x_2$ model', color='blue')
plt.plot(scan_truth, tmu_truth, lw=2, ls="--", color='orange', label="Truth")
plt.axis(ymin=0)
plt.legend()
plt.xlabel(r"$\mu_\mathrm{signal}$")
plt.ylabel(r"$t_\mu$")

## Sufficient statistics and the density ratio

For a mixture of signal and background, the event density can be written as

\begin{equation}
p(x;\mu)
= \frac{\mu\nu_S p_S(x)+\nu_B p_B(x)}{\mu\nu_S+\nu_B}.
\end{equation}

Factoring out $p_B(x)$ shows that the dependence on $x$ relevant for $\mu$ enters through

\begin{equation}
T(x)=\frac{p_S(x)}{p_B(x)}.
\end{equation}

This is the key bridge to neural simulation-based inference: instead of hand-designing a histogram, we can train a neural network to approximate the likelihood ratio.

## A sufficient statistics

\begin{equation}
p(x;\mu) = \frac{1}{\mu\nu_S+\nu_B} \left[\mu \nu_S p_S(x) + \nu_B p_B(x)\right] = p_B(x) \times \frac{1}{\mu\,\nu_S/\nu_B+1} \times \left[ \mu \frac{\nu_S}{\nu_B} \frac{p_S(x)}{p_B(x)} + 1 \right] = h(x)\times g(\mathbf{T}(x);\mu)
\end{equation}

$\mathbf{T}(x) = p_S(x)/p_B(x)$ is a sufficient statistics and can be approximated by a NN

In [ ]:
from utils import split_train_inference

SPLIT_SEED = 0
TRAIN_FRACTION = 0.5  # fraction of each sample used for TRAINING

signal_split, _ = split_train_inference(signal, train_fraction=TRAIN_FRACTION, seed=SPLIT_SEED)
background_split, _ = split_train_inference(background, train_fraction=TRAIN_FRACTION, seed=SPLIT_SEED)

print(f"Training on {len(signal_split):,} signal + {len(background_split):,} background events")

In [ ]:
numerator_hypothesis = signal_split.astype('float32').copy()
denominator_hypothesis = background_split.astype('float32').copy()

In [ ]:
numerator_hypothesis["weights"] = numerator_hypothesis['weight']
numerator_hypothesis["weights_normed"] = numerator_hypothesis['weight'] / numerator_hypothesis['weight'].sum()
numerator_hypothesis["train_labels"] = 1.0

denominator_hypothesis["weights"] = denominator_hypothesis['weight']
denominator_hypothesis["weights_normed"] = denominator_hypothesis['weight'] / denominator_hypothesis['weight'].sum()
denominator_hypothesis["train_labels"] = 0.0

In [ ]:
N_TRAIN = 1_500_000  # None = use all events

training_dataframe = pd.concat([numerator_hypothesis, denominator_hypothesis], ignore_index=True)
training_dataframe = training_dataframe.sample(frac=1, random_state=42, ignore_index=True)

if N_TRAIN is not None:
    training_dataframe = training_dataframe.head(N_TRAIN).reset_index(drop=True)
    # Re-normalize per-class weights so each class still sums to 1 after subsetting
    for label_val in [0.0, 1.0]:
        mask = training_dataframe["train_labels"] == label_val
        training_dataframe.loc[mask, "weights_normed"] = (
            training_dataframe.loc[mask, "weight"]
            / training_dataframe.loc[mask, "weight"].sum()
        )

In [ ]:
ensemble_size = 1

## Classifier as a density-ratio estimator

A binary classifier trained to distinguish signal from background learns a monotonic function of the density ratio.  With appropriate class priors, the classifier score $s(x)$ is related to

\begin{equation}
\frac{p_S(x)}{p_B(x)} \approx \frac{s(x)}{1-s(x)}.
\end{equation}

The details of calibration matter in practice, but the conceptual point is simple: classification can be used as a route to likelihood-ratio estimation.

In [ ]:
from nsbi_common_utils.training import density_ratio_trainer

training_features = ['x1','x2','x3','x4','x5']
trainer = [None for i in range(ensemble_size)]

for ens_index in range(ensemble_size):
    trainer[ens_index] = density_ratio_trainer(
        dataset=training_dataframe,
        weights=training_dataframe["weights_normed"],
        training_labels=training_dataframe["train_labels"],
        features=training_features,
        features_scaling=training_features,
        sample_name=["signal", "reference"],
        output_name="",
        path_to_figures="plots_SigvsBkg/",
        path_to_models="models_SigvsBkg/",
    )

In [ ]:
for ens_index in range(ensemble_size):

    trainer[ens_index].train(
        hidden_layers=3,
        neurons=1024,
        number_of_epochs=5,
        batch_size=4096,
        learning_rate=1e-3,
        scalerType="MinMax",
        ensemble_index=ens_index,
        verbose=1,
        holdout_split=0.25, # validation
        validation_split=0.2,
        callback_patience=10,
        num_workers=4,
        load_trained_models=False,
        calibration=False,
        type_of_calibration = "histogram",
        recalibrate_output=True,
        num_bins_cal = 50
    )

In [ ]:
sig_ratios = []
bkg_ratios = []
for ens_index in range(ensemble_size):
    path_to_saved_scaler = f"models_SigvsBkg/model_scaler{ens_index}.bin"
    path_to_saved_model  = f"models_SigvsBkg/model{ens_index}.onnx"

    scaler, model = load_trained_model(path_to_saved_model, path_to_saved_scaler)

    signal_ratio = nsbi_common_utils.training.utils.predict_with_model(
        signal_split[training_features].astype('float32'),
        scaler=scaler,
        model=model,
    )
    background_ratio = nsbi_common_utils.training.utils.predict_with_model(
        background_split[training_features].astype('float32'),
        scaler=scaler,
        model=model,
    )
    sig_ratios.append(signal_ratio)
    bkg_ratios.append(background_ratio)

## Building a histogram of the learned statistic

After training, the neural network maps each event from the original feature space to a one-dimensional ratio-like statistic.  We then return to the same binned-likelihood machinery as before, but now the bins are placed on a more informative variable.

This keeps the statistical model familiar while replacing the hand-chosen observable with a learned summary.

In [ ]:
bins_ratio = np.linspace(0,14,15)
h_sig_ratio = hist_template(sig_ratios[0], bins_ratio, signal_split.weight)
h_bkg_ratio = hist_template(bkg_ratios[0], bins_ratio, background_split.weight)
plot_histograms(h_sig_ratio,h_bkg_ratio,bins_ratio, True, r'$p_S/p_B$')

In [ ]:
ws_ratio = make_workspace((h_sig_ratio).tolist(), (h_bkg_ratio).tolist(), (h_sig_ratio+h_bkg_ratio).tolist())
model_ratio = nsbi_common_utils.models.sbi_parametric_model(workspace=ws_ratio, measurement_to_fit="meas")
list_params, init_values = model_ratio.get_model_parameters()
num_unconstrained = model_ratio.num_unconstrained_param
inf_model_ratio = nsbi_common_utils.inference.inference(
    model_nll=model_ratio.model,
    initial_values=init_values,
    list_parameters=list_params,
    num_unconstrained_params=num_unconstrained,
    model_grad=model_ratio.model_grad)

In [ ]:
print("\n" + "="*40)
print(" MODEL pS/pB FIT RESULTS ")
print("="*40 + "\n")
inf_model_ratio.perform_fit(freeze_params=[])

In [ ]:
scan_points_model_ratio, NLL_value_model_ratio, scan_points_StatOnly_model_ratio, NLL_value_StatOnly_model_ratio = inf_model_ratio.perform_profile_scan(parameter_name = 'mu',
                             freeze_params = [],
                             bound_range = (0, 3.0),
                             fit_strategy = 0,
                             doStatOnly = True,
                             size = 50)

## Final comparison

The final plot compares several summaries using the same likelihood-scan language:

- $x_1$: simple and interpretable, but lossy;
- $(x_1,x_2)$: more information, but more bins;
- learned $p_S/p_B$: compressed but targeted to the inference task;
- truth ratio: the ideal reference for this toy problem.

The teaching message is that better summaries produce sharper likelihood scans without changing the underlying parameter being measured.

In [ ]:
plt.plot(scan_points_model_x1, NLL_value_model_x1, label=r'$x_1$ model', color='red')
plt.plot(scan_points_model_x1x2, NLL_value_model_x1x2, label=r'$x_1-x_2$ model', color='blue')
plt.plot(scan_points_model_ratio, NLL_value_model_ratio, label=r'$p_S/p_B (x_1,x_2,x_3,x_4,x_5)$ model', color='green')
plt.plot(scan_truth, tmu_truth, lw=2, ls="--", color='orange', label="Truth")
plt.axis(ymin=0)
plt.legend()
plt.xlabel(r"$\mu_\mathrm{signal}$")
plt.ylabel(r"$t_\mu$")

**Challenge**: What can you change to make the NN model closer to the truth?